# GDC Pathology Reports for CDDP_EAGLE-1 Cases in IDC

[CDDP_EAGLE](https://doi.org/10.7937/TCIA.2019.f4mcnm36) is a lung adenocarcinoma cohort (49 patients) with CT and whole-slide images available in IDC (`cddp_eagle_1`).
The GDC hosts a matched pathology report PDF for 48 of the 49 IDC patients (98%).

**Key difference from TCGA reports:** these PDFs are **pure scans** — no embedded text layer.
OCR is required to extract text.
The scans are high quality (600 DPI, ~2 MB each) and OCR with tesseract takes ~1.5 s/page.

**No authentication required** — GDC pathology reports are open-access files.

## Requirements

```bash
# System
brew install tesseract          # macOS
sudo apt install tesseract-ocr  # Ubuntu/Debian

# Python
pip install idc-index requests pymupdf pytesseract
```

In [ ]:
import csv
import io
import json
import os
import re
import tempfile
import time
from collections import Counter, defaultdict
from pathlib import Path

import fitz          # pymupdf
import pytesseract
import requests
from idc_index import IDCClient

GDC_FILES_URL = "https://api.gdc.cancer.gov/files"
GDC_DATA_URL  = "https://api.gdc.cancer.gov/data"

client = IDCClient()
print("IDC version :", client.get_idc_version())
print("tesseract   :", pytesseract.get_tesseract_version())

## 1. Get CDDP Patient IDs from IDC

`PatientID` in IDC equals the GDC `submitter_id` (format: `CDDP-XXXX`).

In [ ]:
df = client.sql_query("""
    SELECT DISTINCT PatientID
    FROM index
    WHERE collection_id = 'cddp_eagle_1'
    ORDER BY PatientID
""")

patient_ids = list(df["PatientID"])
print(f"{len(patient_ids)} patients in cddp_eagle_1")
print("Sample:", patient_ids[:5])

## 2. Query GDC for Pathology Report Metadata

In [ ]:
payload = {
    "filters": json.dumps({
        "op": "and",
        "content": [
            {"op": "in", "content": {"field": "cases.submitter_id", "value": patient_ids}},
            {"op": "=",  "content": {"field": "data_type", "value": "Pathology Report"}},
        ],
    }),
    "fields": "file_id,file_name,cases.submitter_id,file_size,md5sum",
    "size": str(len(patient_ids) + 10),
    "format": "JSON",
}
r = requests.get(GDC_FILES_URL, params=payload, timeout=30)
r.raise_for_status()
reports = r.json()["data"]["hits"]

case_to_report = {h["cases"][0]["submitter_id"]: h for h in reports}

matched   = set(patient_ids) & case_to_report.keys()
unmatched = set(patient_ids) - case_to_report.keys()
print(f"Reports found    : {len(reports)}")
print(f"Patients matched : {len(matched)} / {len(patient_ids)} ({len(matched)/len(patient_ids)*100:.1f}%)")
print(f"Patients missing : {sorted(unmatched)}")

sizes = [h["file_size"] for h in reports]
print(f"\nFile size  min={min(sizes)/1e6:.1f} MB  "
      f"median={sorted(sizes)[len(sizes)//2]/1e6:.1f} MB  "
      f"max={max(sizes)/1e6:.1f} MB")

sample = reports[0]
print(f"\nExample — {sample['cases'][0]['submitter_id']}:")
print(f"  file_name: {sample['file_name']}")
print(f"  file_id  : {sample['file_id']}")
print(f"  download : {GDC_DATA_URL}/{sample['file_id']}")

## 3. Download Reports

Each PDF is ~2 MB. The full collection (48 files) is ~100 MB.

In [ ]:
def download_gdc_file(file_id: str, dest_path: Path) -> Path:
    """Stream a GDC file to disk."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(f"{GDC_DATA_URL}/{file_id}", stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    return dest_path


def download_reports(
    case_to_report: dict,
    output_dir: str | Path,
    limit: int | None = None,
    delay_s: float = 0.3,
) -> list[Path]:
    """
    Download CDDP_EAGLE pathology report PDFs.

    Parameters
    ----------
    case_to_report : dict mapping submitter_id → GDC file-metadata dict
    output_dir     : destination; PDFs saved as {submitter_id}/{file_name}
    limit          : max cases to download (None = all)
    delay_s        : polite pause between requests
    """
    output_dir = Path(output_dir)
    downloaded = []
    items = list(case_to_report.items())
    if limit:
        items = items[:limit]

    for case_id, meta in items:
        dest = output_dir / case_id / meta["file_name"]
        if dest.exists():
            print(f"  skip (exists): {dest.name}")
            downloaded.append(dest)
            continue
        print(f"  downloading {case_id} — {meta['file_size']/1e6:.1f} MB")
        download_gdc_file(meta["file_id"], dest)
        downloaded.append(dest)
        time.sleep(delay_s)

    return downloaded


# Download a sample (3 cases)
paths = download_reports(
    case_to_report,
    output_dir="/tmp/gdc_pathology/cddp_eagle_1",
    limit=3,
)
for p in paths:
    print(f"  {p}  ({p.stat().st_size/1e6:.1f} MB)")

## 4. OCR Text Extraction

CDDP_EAGLE reports are **pure scans** — no embedded text layer.
Every PDF is a single full-page image at 600 DPI (5,100 × 6,600 px).
Tesseract extracts clean text in ~1.5 s/page.

### PDF structure

| Property | Value |
|----------|-------|
| Pages | 1–2 |
| Image resolution | 5,100 × 6,600 px (600 DPI letter) |
| File size | 1.5 – 4.4 MB |
| Text layer | None — OCR required |
| OCR speed (tesseract 5, Apple M-series) | ~1.5 s/page |

### Report sections

Each report contains (in order):
1. **Material description** — tissue specimens labeled A, B, C …
2. **Macro description** — gross dimensions, appearance
3. **Micro description** — histology, WHO classification, grade, margin status, lymph node status
4. **Staging** — `pTNM: pTx Nx` and ICD-10 code
5. **TCGA-style QA footer** — Diagnosis Discrepancy checkboxes (noisy, ignore)

In [ ]:
def ocr_pdf(pdf_path: Path, dpi: int = 300) -> str:
    """
    OCR a scan-only PDF using PyMuPDF + tesseract.

    dpi=300 is a good balance of speed and accuracy for these 600 DPI originals.
    Use dpi=200 for batch speed; dpi=400 for maximum quality.
    """
    doc = fitz.open(pdf_path)
    pages_text = []
    for page in doc:
        mat = fitz.Matrix(dpi / 72, dpi / 72)
        pix = page.get_pixmap(matrix=mat, colorspace=fitz.csGRAY)
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
            tmp = f.name
        pix.save(tmp)
        text = pytesseract.image_to_string(tmp, lang="eng", config="--psm 6")
        os.unlink(tmp)
        pages_text.append(text)
    doc.close()
    return "\n\n".join(pages_text)


# Demo on downloaded samples
for p in sorted(Path("/tmp/gdc_pathology/cddp_eagle_1").rglob("*.pdf")):
    t0 = time.time()
    text = ocr_pdf(p)
    elapsed = time.time() - t0
    doc = fitz.open(p)
    pages = len(doc); doc.close()
    print(f"{p.parent.name}  pages={pages}  {elapsed:.1f}s  {len(text):,} chars")
    print(text[:500].strip())
    print()

## 5. Structured Field Extraction

The OCR'd text is clean enough for regex extraction of key staging fields.
CDDP_EAGLE reports are from a single Italian institution (EAGLE trial) and
follow a consistent template, which makes pattern matching reliable.

In [ ]:
def extract_staging_fields(text: str) -> dict:
    """Best-effort regex extraction from OCR'd CDDP_EAGLE pathology reports."""
    fields = {}

    # TNM — pT*, pN*, pM* (including combined pT2,N0 style)
    tnm = re.findall(r'p[TtNnMm]\d[\w,]*', text)
    if tnm:
        fields["tnm"] = list(dict.fromkeys(tnm))

    # Histologic grade — G1–G4 or 'grade N'
    grade = re.findall(r'\bG\s*[1-4]\b|[Gg]rade\s+[1-4]', text)
    if grade:
        fields["grade"] = list(dict.fromkeys(grade))

    # WHO histology classification line
    who = re.search(
        r'(?:Pulmonary|Lung)[^\n]{0,60}(?:adenocarcinoma|carcinoma|squamous)[^\n]{0,80}',
        text, re.IGNORECASE
    )
    if who:
        fields["histology"] = who.group(0).strip()

    # Tumor 3D size from gross description — e.g. '3.2 x 2.5 x 2 cm'
    sizes_3d = re.findall(
        r'(\d+\.?\d*)\s*[xX×]\s*(\d+\.?\d*)\s*[xX×]\s*(\d+\.?\d*)\s*(cm|mm)',
        text, re.IGNORECASE
    )
    if sizes_3d:
        fields["tumor_size_3d"] = [f"{a}×{b}×{c} {u}" for a, b, c, u in sizes_3d[:2]]

    # Lymph node status — 'N lymph nodes free of metastases'
    ln_free = re.findall(r'(\d+)\s*(?:peribronchial\s*)?lymph\s*nodes?\s*free', text, re.IGNORECASE)
    if ln_free:
        fields["lymph_nodes_free"] = [int(n) for n in ln_free]

    # ICD-10 site code — e.g. C34.1, C34.3
    icd = re.findall(r'\bC3[0-9]\.[0-9]\b', text)
    if icd:
        fields["icd10_site"] = list(dict.fromkeys(icd))

    # Margin status
    if re.search(r'margin[s]?\s*(?:of\s*\w+\s*)?(?:free|negative|clear)', text, re.IGNORECASE):
        fields["margins"] = "negative"

    return fields


for p in sorted(Path("/tmp/gdc_pathology/cddp_eagle_1").rglob("*.pdf")):
    text = ocr_pdf(p)
    fields = extract_staging_fields(text)
    print(f"{p.parent.name}")
    for k, v in fields.items():
        print(f"  {k:20s}: {v}")
    print()

## 6. Batch OCR and Save to Text Files

For larger workflows, OCR all downloaded reports once and save `.txt` sidecars.
Subsequent runs skip already-processed files.

In [ ]:
def ocr_collection(
    pdf_root: str | Path,
    dpi: int = 300,
    force: bool = False,
) -> dict[str, str]:
    """
    OCR all PDFs under pdf_root, saving .txt sidecars next to each PDF.

    Returns dict mapping case_id → extracted text.
    """
    pdf_root = Path(pdf_root)
    results = {}
    pdfs = sorted(pdf_root.rglob("*.pdf"))
    print(f"Found {len(pdfs)} PDFs under {pdf_root}")

    for pdf in pdfs:
        case_id = pdf.parent.name
        txt_path = pdf.with_suffix(".txt")

        if txt_path.exists() and not force:
            text = txt_path.read_text()
            print(f"  {case_id}  (cached, {len(text):,} chars)")
        else:
            t0 = time.time()
            text = ocr_pdf(pdf, dpi=dpi)
            txt_path.write_text(text)
            print(f"  {case_id}  OCR {time.time()-t0:.1f}s → {len(text):,} chars")

        results[case_id] = text

    return results


texts = ocr_collection("/tmp/gdc_pathology/cddp_eagle_1")

## 7. Generate a GDC Download Manifest

For downloading all 48 reports (~100 MB) the [GDC Data Transfer Tool](https://gdc.cancer.gov/access-data/gdc-data-transfer-tool) is more efficient than streaming individually.

In [ ]:
def make_gdc_manifest(reports: list[dict]) -> str:
    """Return manifest TSV for: gdc-client download -m manifest.txt -d ./output"""
    buf = io.StringIO()
    w = csv.writer(buf, delimiter="\t")
    w.writerow(["id", "filename", "md5", "size", "state"])
    for r in reports:
        w.writerow([r["file_id"], r["file_name"], r.get("md5sum", ""), r.get("file_size", ""), "released"])
    return buf.getvalue()


manifest = make_gdc_manifest(reports)
manifest_path = Path("/tmp/gdc_pathology/cddp_eagle_1_manifest.txt")
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(manifest)

print(f"Manifest: {manifest_path} ({len(reports)} files)")
print("\n".join(manifest.splitlines()[:4]))
print(f"\n# gdc-client download -m {manifest_path} -d /path/to/output")